# Train tokenizers from forced-alignment evidence

Use this notebook to build the **force-ranked** and **frequency-ranked** Bigram vocabularies for one OnHW fold-0 split. Both use TVA’s existing greedy Bigram encoder. Each adds 100 bigrams that appear in encoded training labels to the same 60 character and blank IDs.

Choose `wi` (writer-independent) or `wd` (writer-dependent). With `build = False`, the notebook shows the planned commands and checks existing outputs. Set `build = True` to run missing export, scoring, and vocabulary steps. Run from the TVA root in the `tva` environment.

[Config map](configs/thesis/forced_alignment/README.md) · [How forced alignment works](forced_alignment_demo.ipynb)

In [1]:
from argparse import Namespace
import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

import yaml
from IPython.display import Markdown, display
from tva.force_bigram import verify_matched_vocabs

setting = "wi"  # Use "wd" for the writer-dependent split.
build = False   # Set True to build any missing tokenizer artifacts.

config_dir = Path("configs/thesis/forced_alignment") / setting
settings = yaml.safe_load((config_dir / "build_tokenizers.yaml").read_text(encoding="utf-8"))
aligner_config = Path(settings["aligner_config"])
aligner = yaml.safe_load(aligner_config.read_text(encoding="utf-8"))
checkpoint = Path(settings["checkpoint"])
data_dir = Path(aligner["dir_dataset"])
fold = aligner["idx_fold"]
work_dir = Path(settings["dir_work"])
boundary_path = work_dir / "evidence" / "boundaries.jsonl"
ranking_path = boundary_path.parent / "continuity_scores" / "pair_continuity_scores.csv"
vocab_dir = work_dir / "tokenizers"
env = {**os.environ, "PYTHONHASHSEED": "42", "OMP_NUM_THREADS": "1"}

def run_stage(outputs, command):
    if all(path.is_file() for path in outputs):
        print("Using", outputs[0])
        return
    print(shlex.join(command), flush=True)
    if build:
        if any(path.exists() for path in outputs):
            raise FileExistsError("This step has partial outputs. See the setup guide before restarting.")
        subprocess.run([sys.executable, *command[1:]], env=env, check=True)

print(f"{setting.upper()} fold {fold}  ·  {data_dir}")
print("Aligner checkpoint", checkpoint)
if not checkpoint.is_file():
    aligner_run = Path(aligner["dir_work"]) / str(fold)
    if aligner_run.exists():
        print("The aligner run directory exists. Wait for its final checkpoint.")
    else:
        print(f"Train the aligner with python main.py --config {aligner_config}")
    if build:
        raise FileNotFoundError(f"The build requires the final checkpoint at {checkpoint}.")

WI fold 0  ·  data/tva/onhw_words500_wi_word_rh
Aligner checkpoint results/thesis/alignment_models/A0_char_wi_rh_unidirectional/0/checkpoints/299.pth


## 1. Export the training boundaries

The saved A0 character model places the known letters on the sensor timeline. The exporter writes one row per adjacent-letter gap, including the force readings and alignment checks.

In [2]:
run_stage(
    [boundary_path, boundary_path.with_name("summary.json")],
    [
        "python", "-m", "tva.forced_alignment.export_boundaries", "--config", str(aligner_config),
        "--checkpoint", str(checkpoint), "--split", "train",
        "--device", settings["device"], "--num-workers", str(settings["num_workers"]),
        "--output", str(boundary_path),
    ],
)

Using results/thesis/forced_alignment_review/bigram_matched_fold0/wi/evidence/boundaries.jsonl


## 2. Rank the usable letter pairs

The scorer combines the local and whole-gap force measurements. It keeps pairs with enough usable observations and writer coverage. The frequency side uses this same candidate pool.

In [3]:
run_stage(
    [ranking_path, ranking_path.with_name("continuity_score_summary.json")],
    ["python", "-m", "tva.forced_alignment.score_continuity", "--input", str(boundary_path)],
)

Using results/thesis/forced_alignment_review/bigram_matched_fold0/wi/evidence/continuity_scores/pair_continuity_scores.csv


## 3. Build both vocabularies

The builder ranks the shared candidates once by force and once by training-text frequency. It replaces selected bigrams that the greedy encoder never emits and checks that every training label round-trips. The two outputs are standard TVA tokenizer JSON files.

In [4]:
run_stage(
    [vocab_dir / condition / f"{fold}.json" for condition in ("frequency", "force")] +
    [vocab_dir / "summary.json", vocab_dir / "run_manifest.json"],
    [
        "python", "-m", "tva.forced_alignment.build_bigram_v0", "--match-emitted",
        "--train", str(data_dir / "train.json"),
        "--char-vocab", str(Path(aligner["dir_tokenizer"]) / f"{fold}.json"),
        "--force-ranking", str(ranking_path), "--output-dir", str(vocab_dir),
    ],
)

Using results/thesis/forced_alignment_review/bigram_matched_fold0/wi/tokenizers/frequency/0.json


## 4. Check the match and find the training configs

Both recognition configs use `main.py`, the standard TVA entry point. Compare the YAML files side by side. This notebook prints the training commands so you can run them one at a time.

In [5]:
conditions = ("frequency", "force")
training_paths = {name: config_dir / f"{name}.yaml" for name in conditions}
configs = {name: yaml.safe_load(path.read_text(encoding="utf-8"))
           for name, path in training_paths.items()}
differences = [key for key in configs["force"]
               if configs["frequency"][key] != configs["force"][key]]
assert set(differences) == {"dir_tokenizer", "dir_work"}
assert all(Path(cfg["dir_tokenizer"]).parent == vocab_dir for cfg in configs.values())
assert all(cfg["idx_fold"] == fold and Path(cfg["dir_dataset"]) == data_dir
           for cfg in configs.values())

if (vocab_dir / "run_manifest.json").is_file():
    verify_matched_vocabs(Namespace(**configs["force"]))
    summary = json.loads((vocab_dir / "summary.json").read_text(encoding="utf-8"))
    lines = ["| Vocabulary | Output classes | Bigram types emitted | Training tokens |",
             "| --- | ---: | ---: | ---: |"]
    for name in conditions:
        info = summary["conditions"][name]
        lines.append(f"| {name} | {info['vocab_size_including_blank']} | "
                     f"{info['distinct_bigram_tokens_used']} | {info['tokens']:,} |")
    display(Markdown("\n".join(lines)))
else:
    print("Build the vocabularies before running recognition.")

for path in training_paths.values():
    print(f"python main.py --config {path}")

| Vocabulary | Output classes | Bigram types emitted | Training tokens |
| --- | ---: | ---: | ---: |
| frequency | 160 | 100 | 70,841 |
| force | 160 | 100 | 75,450 |

python main.py --config configs/thesis/forced_alignment/wi/frequency.yaml
python main.py --config configs/thesis/forced_alignment/wi/force.yaml
